In [48]:
# Tokenization & HF Dataset Preparation ==================================
from datasets import Dataset
from transformers import AutoTokenizer
# Train/Val Split ========================================================
from sklearn.model_selection import train_test_split
# Utils ==================================================================
from src.data import load_data

In [49]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

In [50]:
INPUT_PATH = "../data/raw/raw-data.csv"
# Safely load the dataframe using my robust function (src=>data[module])
df = load_data(file_path=INPUT_PATH)

Successfully loaded data from raw-data.csv


In [51]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

In [52]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [53]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Train dataset size: {len(val_dataset)}")

def preprocess(examples):
    # Format each sample into an instruction prompt and tokenize it safely.
    prompts = []

    # Safely iterate and convert everything to string to prevent type errors
    for history, action_id, response in zip(examples["history"], examples["action_id"], examples["response"]):
        h_text = str(history) if history is not None else ""
        r_text = str(response) if response is not None else ""
        a_id = str(action_id) if action_id is not None else ""

        prompt = (
            f"### Instruction:\n"
            f"You are a skilled debater. Respond to the opponent's argument using debate strategy ID {action_id}.\n\n"
            f"### Opponent Argument:\n"
            f"{history}\n\n"
            f"### Response:\n"
            f"{response}{tokenizer.eos_token}"
        )
        prompts.append(prompt)

    tokenized = tokenizer(
        prompts,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors=None
    )

    # For Causal Language Modeling, labels are identical to input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

Train dataset size: 799
Train dataset size: 200


In [54]:
tokenized_train = train_dataset.map(
    preprocess,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess,
    batched=True,
    remove_columns=val_dataset.column_names
)

print("\n--- Preprocessing Complete ---")
print(f"Tokenized train size: {len(tokenized_train)}")
print(f"Tokenized val size: {len(tokenized_val)}")

Map: 100%|██████████| 200/200 [00:00<00:00, 1635.56 examples/s]


--- Preprocessing Complete ---
Tokenized train size: 799
Tokenized val size: 200


# ⚙️ Data Preprocessing & Tokenization

---

### 1. Train / Validation Split
* **Split Ratio:** 80% training (`799` samples) and 20% validation (`200` samples)
* **Configuration:** Randomized split using a fixed random seed (`random_state=42`) to ensure reproducibility

---

### 2. Instruction Prompt Formatting
* **Template Structure:** Formatted for Supervised Fine-Tuning (SFT) using a structured prompt template:
    * **System/Instruction:** Defines the model's role as a skilled debater using a specific strategy ID (`action_id`)
    * **Opponent Argument:** Injects the debate context (`history`)
    * **Response:** Appends the target response followed by the model's end-of-sequence token (`tokenizer.eos_token`)

---

### 3. Tokenization & Padding Settings
* **Base Tokenizer:** `Qwen/Qwen2.5-7B-Instruct` tokenizer
* **Padding Configuration:** Right-side padding (`padding_side="right"`) with `pad_token` mapped to `eos_token`
* **Length Constraints:** Truncated and padded to a maximum sequence length of `512` tokens
* **Labels Setup:** Configured `labels` identical to `input_ids` for standard causal language modeling loss computation